# Session 10 — The model landscape: choosing a model

**Catalog · cost · speed · quality · the selection matrix**

The loan assistant does three different jobs. Each needs a different model. Your task: fill in the evidence, then choose one model per job and defend it.

Run top to bottom (Kernel → Restart & Run All). Before starting: `git pull`, `uv sync`, then `uv run jupyter lab module01/s10_model_matrix.ipynb`.

**How the decision is made (memorise this order):** 1) **hard constraints** eliminate models (data must stay in-house, latency limit, context too small); 2) among survivors, a **weighted score** of cost, speed and quality ranks them; 3) you **verify** the winner on your own data (S17 gives you the tool).


## 1. Setup

`landscape_utils.py` holds the catalog, the cost formula, the constraint filter and the scorer. `tokens_utils.py` (from S9b) gives us the two clients: Groq from `.env`, and Ollama on your laptop.


In [1]:
import os
import sys

sys.path[:0] = [".", "module01", "../module01"]

from landscape_utils import *
from tokens_utils import chat_client, ollama_client

ollama = ollama_client()
# Fallback handling: If GROQ_API_KEY is missing or invalid in .env, we gracefully set groq to None
# so the notebook can still run all local/Ollama and analytical portions without crashing.
try:
    groq = chat_client()  # Groq, from .env
except (
    Exception  # noqa: BLE001
) as e:  # no key yet: the notebook still runs; the hosted rows are just skipped
    groq = None
    print("no hosted client:", str(e)[:80])
GROQ_MODEL = os.getenv("MODEL")
print(
    "hosted model:",
    GROQ_MODEL,
    "| expected EMI:",
    EXPECTED_EMI,
    "| expected ratio:",
    EXPECTED_RATIO,
    "%",
)


hosted model: openai/gpt-oss-20b | expected EMI: 17594.09 | expected ratio: 42.7 %


## 2. The catalog — ten rows, one table

Read `models_catalog.json` as a table. **Every value is indicative** (September 2026) — the price tiers in particular come from third-party trackers. `quality` and `latency_s` are the instructor's _starting guesses_; you will replace two of them with measurements in section 4.

Terms in the columns: **open-weight** = the weights can be downloaded; **runs_on_prem** = you can run it where the data lives; **ctx_tokens** = the context window; **price_in / price_out** = USD per million tokens; **reasoning** = spends hidden tokens thinking first.


In [2]:
cat = load_catalog()
print(
    f"{'model':46} {'access':11} {'open':5} {'ctx':>8} {'in$':>6} {'out$':>6} {'Q':>2} {'s':>4}"
)
# Render the baseline catalog in a compact tabular view to compare pricing, context,
# and initial instructor estimates for quality and latency before taking empirical measurements.
for m in cat:
    pin = "?" if m["price_in"] is None else f"{m['price_in']:.3f}"
    pout = "?" if m["price_out"] is None else f"{m['price_out']:.2f}"
    print(
        f"{m['name'][:46]:46} {m['access']:11} {str(m['open_weight'])[:1]:5} {m['ctx_tokens']:8d} {pin:>6} {pout:>6} {m['quality']:2d} {m['latency_s']:4.1f}"
    )


model                                          access      open       ctx    in$   out$  Q    s
gpt-oss-20b on Groq                            hosted-api  T       131072  0.075   0.30  3  1.5
gpt-oss-120b on Groq                           hosted-api  T       131072  0.150   0.60  4  2.0
Qwen3 32B on Groq                              hosted-api  T       131072  0.290   0.59  3  2.0
Llama 4 Scout 17B-16E on Groq                  hosted-api  T       131072  0.110   0.34  3  1.5
Llama 3.1 8B Instant on Groq                   hosted-api  T       131072  0.050      ?  2  1.0
llama3.2:3b on your laptop (Ollama)            local       T       131072  0.000   0.00  2  8.0
gpt-oss-20b self-hosted on a smaller GPU serve self-hosted T       131072  0.000   0.00  3  2.5
gpt-oss-120b self-hosted on your own GPU serve self-hosted T       131072  0.000   0.00  4  3.0
Frontier flagship (closed, vendor API)         vendor-api  F       200000  5.000  25.00  5  6.0
Frontier mid-tier (closed, vendor API)  

**Exercise 1 — fill the blank.** One row has `?` for output price. Look it up on the provider's pricing page and put it in the dictionary below (this is a real skill: catalogs go stale).

Ans: Checked Groq pricing page for Llama 3.1 8B Instant; found it was deprecated for developer tier in August 2026 and moved to contact sales/enterprise contracts, with Groq directing users to gpt-oss-20b.

**Exercise 2 — verify two rows.** Pick two rows, open the provider's page, and note whether the price and context window still match. Write what you found in section 6.

_Question 1 — Why is the self-hosted row's price 0 per token but its monthly cost not zero?_

**Ans:** The monthly cost could be the electricity bill, lease for the GPU, or the datcenter coolinf, rack space, and the salaries of the engineers keeping the server alive.


In [ ]:
# Overrides dictionary: Allows patching outdated or missing catalog values (e.g. unknown output pricing)
# without mutating the underlying JSON catalog file.
OVERRIDES = {
    "groq-llama31-8b": {
        "price_out": 0.08
    },  # <- exercise 1: replace ??? with the number you found, this model was depreciated in Agust 16 this year.
}


## 3. Three jobs, three bills

One product, three scenarios with different volumes and different rules. `SCENARIOS` holds them; edit any number to test "what if".

- **faq** — public product questions. Cheap and high-volume; no customer data.
- **documents** — reads a customer's salary slips and statements. **Customer PII: data must stay in-house.**
- **agent_assist** — live suggestions to a call-centre agent. **Must answer within about 3 seconds.**


In [4]:
# Project monthly run-rate per scenario: Multiplies daily call volume and token counts
# against per-million token rates (or fixed hardware amortisation for self-hosted instances)
# to reveal dramatic cost differences across architectures.
for key, sc in SCENARIOS.items():
    print(
        f"\n{key}: {sc['title']}  ({sc['calls_per_day']:,} calls/day, {sc['in_tokens']:,} in / {sc['out_tokens']} out per call)"
    )
    for m in cat:
        m = {**m, **OVERRIDES.get(m["id"], {})}
        inr = monthly_cost_inr(m, sc)
        print(
            f"   {m['name'][:46]:46} "
            + ("price unknown" if inr is None else f"Rs {inr:>12,}/month")
        )



faq: Public product FAQ chatbot  (20,000 calls/day, 1,200 in / 200 out per call)
   gpt-oss-20b on Groq                            Rs        8,460/month
   gpt-oss-120b on Groq                           Rs       16,920/month
   Qwen3 32B on Groq                              Rs       26,282/month
   Llama 4 Scout 17B-16E on Groq                  Rs       11,280/month
   Llama 3.1 8B Instant on Groq                   Rs        4,286/month
   llama3.2:3b on your laptop (Ollama)            Rs            0/month
   gpt-oss-20b self-hosted on a smaller GPU serve Rs       65,800/month
   gpt-oss-120b self-hosted on your own GPU serve Rs      188,000/month
   Frontier flagship (closed, vendor API)         Rs      620,400/month
   Frontier mid-tier (closed, vendor API)         Rs      248,160/month
   Frontier economy tier (closed, vendor API)     Rs       62,040/month

documents: Loan-document analysis (customer PII)  (2,000 calls/day, 6,000 in / 500 out per call)
   gpt-oss-20b on Groq      

_Question 2 — In `faq`, the flagship tier costs how many times more than gpt-oss-20b per month? In which scenario does that gap matter least, and why?_

**Ans:** Frontier Flagship tier costs 73 times more than gpt-oss-20b per month. The gap matters least in the documents scenario. Volume here is the lowest, the bussiness priority is dominated by accuracy and compliance(quality is weighted at 60%, cost at only 20%).


## 4. Measure — the same prompt on a hosted model and on your laptop

Catalog numbers are a prior; **your measurements are evidence.** We send one prompt to Groq and to Ollama, time it, and check the arithmetic.

- **TTFT** — time to first token you can see. For a reasoning model this includes the hidden thinking.
- **tokens/s** — output speed.
- The prompt has a checkable answer, so `check_answer` tells us whether the model got the EMI and the EMI-to-income ratio right. That is a tiny **evaluation**; S17 builds the full version.


In [5]:
print(PROMPT, "\n")
RESULTS = {}
# Empirical benchmarking: Catalog numbers are only priors—we measure actual TTFT (time-to-first-token),
# token generation speed, and verify domain accuracy (financial arithmetic) on live models.
for label, client, model in [
    ("groq", groq, GROQ_MODEL),
    ("ollama", ollama, "llama3.2:3b"),
]:
    if client is None:
        print(f"--- {label}: skipped (no client - add your key to .env)\n")
        continue
    try:
        r = time_stream(client, model)
        r["check"] = check_answer(r["text"])
        RESULTS[label] = r
        print(f"--- {label}: {model}")
        print(
            f"first token {r['ttft_s']} s | total {r['total_s']} s | {r['completion_tokens']} tokens | {r['tokens_per_s']} tokens/s"
        )
        print("arithmetic check:", r["check"])
        print(r["text"].strip()[:500], "\n")
    except Exception as e:  # noqa: BLE001
        print(f"--- {label}: skipped ({type(e).__name__}: {str(e)[:120]})\n")


Applicant earns Rs 60,000 per month and already pays Rs 8,000 per month in EMIs. She asks for a Rs 8,00,000 personal loan at 11.5% per annum for 60 months. Compute the monthly EMI, then say whether (EMI + existing EMIs) is under 50% of her income. Answer in three short lines. 

--- groq: openai/gpt-oss-20b
first token 1.71 s | total 1.78 s | 869 tokens | 489.2 tokens/s
arithmetic check: {'emi_ok': True, 'ratio_ok': False, 'score_out_of_2': 1}
- EMI ≈ ₹17,600 per month.  
- Total debt service (₹17,600 + ₹8,000) ≈ ₹25,600.  
- ₹25,600 is below 50 % of her ₹60,000 income. 

--- ollama: llama3.2:3b
first token 7.16 s | total 8.54 s | 111 tokens | 13.0 tokens/s
arithmetic check: {'emi_ok': False, 'ratio_ok': False, 'score_out_of_2': 0}
To compute the monthly EMI, we need to calculate the principal amount, interest rate, and loan tenure in monthly terms. 

Principle = 800000, Rate = 11.5%/year / 12 = 0.009583/month, Tenure = 5 years/12 = 60 months

After calculation, the EMI will be Rs 18,43

**Score the answers yourself** with the rubric, then put your numbers in the overrides (they replace the catalog guesses for these two rows):


In [6]:
print(RUBRIC)


Quality rubric (1-5), fill in yourself after reading the answer:
    5  EMI and ratio both right, exactly three short lines, clear yes/no on the 50% test
    4  Both numbers right but format or wording off
    3  One number right, or right numbers with a wrong conclusion
    2  Both numbers wrong but the method is visible
    1  Wrong and confident, or no answer


`groq: openai/gpt-oss-20b`: EMI is rounded correctly (~₹17,600). Total repayment is calculated correctly (₹25,600). The 50% conclusion is correct (₹25,600 < ₹30,000). Exactly three short lines. However, it omitted the explicit ratio percentage ("42.7%").

Grade: 4

`ollama: llama3.2:3b`: Did NOT follow the "three short lines" instruction (wrote 8 lines). Both EMI (₹12,939) and ratio (40%) are mathematically wrong. BUT the formula and method are clearly visible.

Grade: 2


In [ ]:
# Feed empirical evidence back into the decision matrix: Replace catalog guestimates
# with real latency measurements and human-evaluated quality scores (rubric 1-5).
MEASURED = {
    "groq-gpt-oss-20b": {
        "latency_s": round(RESULTS["groq"]["total_s"], 1),
        "quality": 4,
    },
    "ollama-llama32-3b": {
        "latency_s": round(RESULTS["ollama"]["total_s"], 1),
        "quality": 2,
    },
}
OVERRIDES.update(MEASURED)
OVERRIDES


{'groq-llama31-8b': {'price_out': 0.08},
 'groq-gpt-oss-20b': {'latency_s': 1.8, 'quality': 4},
 'ollama-llama32-3b': {'latency_s': 8.5, 'quality': 2}}

_Question 3 — Which model was faster to the first token, and which produced more tokens per second? Were they the same one? What does that tell you about "fast"?_

**Ans:** `groq-gpt-oss-20b`(2.18s vs 2.4s) was faster to the first token. And `groq-gpt-oss-20b`(567.2 tok/s vs 31.2 tok/s) produced more tokens per second. They were the same one. This tells my spider senses that my laptop is tras- sorry it tells me that Groq maintains a more high end hardware, which is their premium LPU chips, compared to me.

Now I think 'fast' here means, this is just a pay-to-win rigged match, Groq is a big company that has their own high end servers, chips(Their LPUs), thier engineers to maintain the data centers, so it's obvious that theirs is 'fast'. Anyways 'Fast' here is two things:

1. Responsiveness(Time to First Token).
2. Throughput(Tokens per Second).

- For chat/UI apps, TTFT matters most so the human isn't staring at a blank screen.
- For background batch jobs, Throughput matters most to chew through millions of rows.


## 5. The selection matrix

For each scenario: **hard constraints first** (a model that fails one is out, however cheap), then a weighted score of cost, speed and quality. Weights are in `SCENARIOS`; change them and watch the winner move.


In [ ]:
# Two-stage selection pipeline:
# 1. Hard constraints filter first (data residency/on-prem, max latency, min quality, daily throughput capacity).
#    Any model that violates a hard requirement is immediately disqualified regardless of price.
# 2. Weighted scoring ranks survivors based on scenario priorities (trade-offs between cost, latency, and quality).
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)

    w_str = f"Cost: {int(sc['weights']['cost'] * 100)}% | Latency: {int(sc['weights']['latency'] * 100)}% | Quality: {int(sc['weights']['quality'] * 100)}%"
    header_title = f"{key.upper()} — {sc['title']}"

    print("\n" + "=" * 98)
    print(f" 🎯 SCENARIO: {header_title}")
    print(f" ⚖️  WEIGHTS : {w_str}")
    print("=" * 98)

    # 1. Qualified & Ranked Table
    print("\n🏆 ELIGIBLE CANDIDATES (Ranked by Weighted Score):")
    if not ranked:
        print("   ⚠️  No models passed the hard constraints.")
    else:
        print(
            f"   {'#':<3} {'MODEL NAME':<48} {'SCORE':>7} {'MONTHLY BILL':>16} {'LATENCY':>10} {'QUALITY':>9}"
        )
        print(f"   {'-' * 3} {'-' * 48} {'-' * 7} {'-' * 16} {'-' * 10} {'-' * 9}")
        for i, m in enumerate(ranked, 1):
            badge = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f"{i}."
            print(
                f"   {badge:<3} {m['name']:<48} "
                f"{m['score']:>7.1f} "
                f"{f'₹{m["cost_inr"]:,}':>15} "
                f"{m['latency_s']:>8.1f} s "
                f"{f'Q{m["quality"]}/5':>9}"
            )
    # 2. Disqualified / Excluded Table
    if excluded:
        print(
            f"\n🚫 DISQUALIFIED MODELS ({len(excluded)} eliminated by hard constraints):"
        )
        print(f"   {'MODEL NAME':<48} │ {'REASON FOR ELIMINATION'}")
        print(f"   {'-' * 48}─┼─{'-' * 47}")
        for name, why in excluded:
            reasons = "; ".join(why)
            print(f"   {name:<48} │ {reasons}")

    print("-" * 98)


 🎯 SCENARIO: FAQ — Public product FAQ chatbot
 ⚖️  WEIGHTS : Cost: 60% | Latency: 20% | Quality: 20%

🏆 ELIGIBLE CANDIDATES (Ranked by Weighted Score):
   #   MODEL NAME                                         SCORE     MONTHLY BILL    LATENCY   QUALITY
   --- ------------------------------------------------ ------- ---------------- ---------- ---------
   🥇   Llama 3.1 8B Instant on Groq                        85.0          ₹4,286      1.0 s      Q2/5
   🥈   gpt-oss-20b on Groq                                 83.7          ₹8,460      1.8 s      Q4/5
   🥉   Llama 4 Scout 17B-16E on Groq                       76.4         ₹11,280      1.5 s      Q3/5
   4.  gpt-oss-120b on Groq                                74.6         ₹16,920      2.0 s      Q4/5
   5.  Qwen3 32B on Groq                                   64.3         ₹26,282      2.0 s      Q3/5
   6.  Frontier economy tier (closed, vendor API)          55.9         ₹62,040      1.5 s      Q3/5
   7.  gpt-oss-20b self-hosted on a s

In [ ]:
# 1. Grab a copy of the faq scenario
faq_test = dict(SCENARIOS["faq"])

# 2. Change the weights: Quality = 0.7, Cost = 0.1, Latency = 0.2
faq_test["weights"] = {"cost": 0.1, "latency": 0.2, "quality": 0.7}

# 3. Re-run rank with the new weights
ranked_new, _ = rank(faq_test, overrides=OVERRIDES)

# 4. Print the new top 3 winners!
for i, m in enumerate(ranked_new[:3], 1):
    print(
        f"{i}. {m['name']} | Score: {m['score']} | Cost: ₹{m['cost_inr']:,} | Q: {m['quality']}"
    )

1. gpt-oss-20b on Groq | Score: 77.9 | Cost: ₹8,460 | Q: 4
2. gpt-oss-120b on Groq | Score: 75.8 | Cost: ₹16,920 | Q: 4
3. Frontier flagship (closed, vendor API) | Score: 70.0 | Cost: ₹620,400 | Q: 5


_Question 4 — In `documents`, what did the hard constraint remove, and what quality did you give up to keep the data in-house? What would you do about it?_

**Ans:** In docs, the hard constraint disqualified 9 cloud models that store the user data in their own data centers. We gave great reasoning beasts for keep this result. Well if we want an alternative, then we should also create a corp like OpenAI, Anthropic, or Google and develop our own Frontier models. Well...... back to reality, since that is nearly impossible we can only maintain our own high end data center with hardware comparable to those frontier models and run open weights for ourselves.

_Question 5 — Change the `faq` weights to quality 0.7, cost 0.1. Does the winner change? What does that tell you about who should choose the weights?_

**Ans:** Yes, the leaderboard changes. gpt-oss-20b is the winner in this scenario because high cost is barely penalized anymore. This teaches us the engineers shouldn't choose weights alone. The bussiness/product leadership must choose the weights. Because in the end they are the who decide the rules and we(engineers) just create the tools areound those rules.


## 6. Your decision — this is what you submit

For each scenario write: the model you would choose, the runner-up, and **what evidence would change your mind**. Also write what you found when you verified two catalog rows (exercise 2), and answers to questions 1–5.

### faq

**Ans:** I would choose `Llama 3.1 8B Instant on Groq` (Deprecated) for FAQs because... well, it kind of won the tournament in the previous cell. It passed the hard constraints easily, its bill is cheap (exactly what we need for something like FAQs), and its speed is also the fastest of the bunch. Its quality doesn't matter as much here because of the nature of FAQs, which receive a large number of prompts a day and only require short and concise answers for each; no deep reasoning is required.

For the runner-up, I would choose `gpt-oss-20b on Groq` because... again, take a look at the leaderboard. What would change my mind? Something even cheaper or faster.

**Stats for Nerds:**

- Winner (Llama 3.1 8B Instant on Groq):
  - Score: 85.0
  - Monthly Bill: ₹4,286/month
  - Latency: 1.0 s
  - Quality: Q2/5
- Runner-up (gpt-oss-20b on Groq):
  - Score: 82.1
  - Monthly Bill: ₹8,460/month
  - Latency: 2.2 s
  - Quality: Q4/5

### documents

**Ans:** I would choose `gpt-oss-20b self-hosted on a smaller GPU server` for the docs scenario, because it won in the table above. This scenario doesn't have much requirement for speed. Its hard constraint is for it to be on-premises. So we chose a self-hosted server.

As for a runner-up, we have `gpt-oss-120b self-hosted on your own GPU server`. The only reason it was a runner-up instead of a winner is because it is extremely expensive (nearly 3× higher!). Its quality is better than the former by one point. This would convince me if it were cheaper to maintain.

**Stats for Nerds:**

- Winner (gpt-oss-20b self-hosted on a smaller GPU server):
  - Score: 70.0
  - Monthly Bill: ₹65,800/month
  - Latency: 2.5 s
  - Quality: Q3/5
- Runner-up (gpt-oss-120b self-hosted on your own GPU server):
  - Score: 45.0
  - Monthly Bill: ₹188,000/month
  - Latency: 3.0 s
  - Quality: Q4/5

### agent_assist

**Ans:** I'd choose `Llama 4 Scout 17B-16E on Groq` for the `agent_assist` scenario, because the leaderboard says so. It has the fastest response time of the bunch, which is crucial for this scenario. Its quality is enough for this task; no need for reasoning. Although this was deprecated, and Groq is suggesting customers migrate to `openai/gpt-oss-20b`.

For the runner-up, I'd choose `gpt-oss-20b on Groq` because it is also pretty fast, and it is cheap.

**Stats for Nerds:**

- Winner (Llama 4 Scout 17B-16E on Groq):
  - Score: 82.6
  - Monthly Bill: ₹18,274/month
  - Latency: 1.5 s
  - Quality: Q3/5
- Runner-up (gpt-oss-20b on Groq):
  - Score: 69.2 (Rank #4 overall, but #1 on pure cost at ₹13,324/mo)
  - Monthly Bill: ₹13,324/month
  - Latency: 2.2 s
  - Quality: Q4/5

**Exercise 1**:
Checked Groq pricing page for Llama 3.1 8B Instant; found it was deprecated for developer tier in August 2026 and moved to contact sales/enterprise contracts, with Groq directing users to gpt-oss-20b. So I used the old 0.08 price for the OVERRIDES.

**Verified rows (exercise 2):**

1. **`groq-llama31-8b` (Llama 3.1 8B Instant on Groq):**
   - **Status:** Deprecated on Groq's free/developer tier on August 16, 2026; moved to committed-spend enterprise contracts ("Contact Sales").
   - **Recommendation:** Groq officially recommends migrating to `openai/gpt-oss-20b`.
   - **Baseline Match:** Context remains 131,072; output pricing is no longer publicly listed for developers.
2. **`groq-qwen3-32b` & `groq-llama4-scout` on Groq:**
   - **Status:** Both deprecated on July 17, 2026 for free/developer tiers.
   - **Recommendation:** Groq points users to `openai/gpt-oss-120b` and `qwen/qwen3.6-27b`.
3. **`gpt-oss-20b` & `gpt-oss-120b` on Groq:**
   - **Status:** Active and stable.
   - **Baseline Match:** Pricing ($0.075 / $0.30 and $0.150 / $0.60 per 1M tokens) and 131,072 context window exactly match the catalog. This also validates the context window for the self-hosted variants.
4. **Frontier Models:**
   - Kept generic across flagship, mid-tier, and economy tiers to reflect broad market rates rather than single-vendor lock-in.

**Q1–Q5:**

_Question 1 — Why is the self-hosted row's price 0 per token but its monthly cost not zero?_

**Ans:** The monthly cost could be the electricity bill, lease for the GPU, or the datacenter cooling, rack space, and the salaries of the engineers keeping the server alive.

_Question 2 — In `faq`, the flagship tier costs how many times more than gpt-oss-20b per month? In which scenario does that gap matter least, and why?_

**Ans:** Frontier Flagship tier costs 73 times more than gpt-oss-20b per month. The gap matters least in the documents scenario. Volume here is the lowest, and the business priority is dominated by accuracy and compliance (quality is weighted at 60%, cost at only 20%).

_Question 3 — Which model was faster to the first token, and which produced more tokens per second? Were they the same one? What does that tell you about "fast"?_

**Ans:** `groq-gpt-oss-20b` (2.18s vs 2.4s) was faster to the first token. And `groq-gpt-oss-20b` (567.2 tok/s vs 31.2 tok/s) produced more tokens per second. They were the same one. This tells my spider senses that my laptop is tras- sorry it tells me that Groq maintains higher-end hardware, which is their premium LPU chips, compared to me.

Now I think "fast" here means this is just a pay-to-win rigged match: Groq is a big company that has its own high-end servers, chips (their LPUs), and their engineers to maintain the data centers, so it's obvious that theirs is "fast". Anyways, "fast" here is two things:

1. Responsiveness (Time to First Token).
2. Throughput (Tokens per Second).

- For chat/UI apps, TTFT matters most so the human isn't staring at a blank screen.
- For background batch jobs, Throughput matters most to chew through millions of rows.

_Question 4 — In `documents`, what did the hard constraint remove, and what quality did you give up to keep the data in-house? What would you do about it?_

**Ans:** In docs, the hard constraint disqualified 9 cloud models that store user data in their own data centers. We gave up great reasoning beasts to keep this result. Well, if we want an alternative, then we should also create a corp like OpenAI, Anthropic, or Google and develop our own Frontier models. Well...... back to reality, since that is nearly impossible, we can only maintain our own high-end data center with hardware comparable to those frontier models and run open weights for ourselves.

_Question 5 — Change the `faq` weights to quality 0.7, cost 0.1. Does the winner change? What does that tell you about who should choose the weights?_

**Ans:** Yes, the leaderboard changes. `gpt-oss-20b` is the winner in this scenario because high cost is barely penalized anymore. This teaches us that engineers shouldn't choose weights alone. Business and product leadership must choose the weights, because in the end they are the ones who decide the rules and we (engineers) just create the tools around those rules.


In [20]:
from pathlib import Path

import landscape_utils

# Persist the final decision matrix, exclusions, and test telemetry to Markdown
# so results can be verified and submitted alongside your architectural defense.
OUT = Path(landscape_utils.__file__).with_name(
    "s10_matrix_output.md"
)  # next to the helper, whichever folder Jupyter runs in
lines = [
    "# Session 10 - selection matrix output\n",
    f"expected EMI {EXPECTED_EMI}, ratio {EXPECTED_RATIO}%\n",
]
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    lines.append(f"\n## {key}: {sc['title']}")
    for i, m in enumerate(ranked, 1):
        lines.append(
            f"{i}. {m['name']} - score {m['score']} - Rs {m['cost_inr']:,}/month - {m['latency_s']} s - Q{m['quality']}"
        )
    for name, why in excluded:
        lines.append(f"- excluded: {name} ({'; '.join(why)})")
lines.append(
    "\n## Measured\n"
    + str(
        {
            k: {kk: v[kk] for kk in ("ttft_s", "total_s", "tokens_per_s", "check")}
            for k, v in RESULTS.items()
        }
    )
)
OUT.write_text("\n".join(lines), encoding="utf-8")
print("saved", OUT)


saved d:\Projects\genai-course-work\module01\s10_matrix_output.md
